# ECMWF week-4 reforecasts
Install `.[ecmwf,notebooks]` in this kernel and configure ECMWF Web API authentication outside the notebook. Accept S2S terms in the ECMWF catalogue first.

`REFERENCE_DATES=None` automatically reads IRI's model-cycle catalogue and the issuance coordinates shared by control and perturbed temperature. Allow `iridl.ldeo.columbia.edu` for discovery, in addition to the ECMWF API/download destinations. If metadata requires IRI authentication/terms, discovery stops with an error; it never substitutes guessed calendar dates. An explicit list of known issuance dates bypasses IRI discovery. `CATALOGUE_CYCLES=None` discovers cycles; use a list such as `["CY48", "CY49"]` to restrict them.

For each ERA5 target week, the planner picks exactly one **newest eligible associated issuance** with the same initialization month/day and coverage in its preceding 20 years. `AS_OF` caps the issuance date (None uses today's UTC date). This is retrospective calibration data, not a historical real-time backtest: model vintages can be newer than the target weather. References/model cycles remain in the output.

`DRY_RUN=True` downloads metadata only and shows the selection, status and reason for every target week. `matched=True` means planned, not that all daily member fields have been verified. Unmatched weeks stay unmatched; recent initializations may require future reforecasts or a separate operational-forecast workflow. Set `DRY_RUN=False` to retrieve and validate the planned fields. Metadata is cached for 24 hours; `REFRESH_CATALOGUE=True` reloads it.

Each output contains weekly mean temperature in Kelvin for the control plus ten perturbed members. Daily GRIB fields are retained. Values are bilinearly interpolated from 1.5° onto the ERA5 coordinates, not physically downscaled. `LEAD_DAYS=26` targets exactly [initialization + 26 days, + 33 days) UTC. Check ERA5 boundary-week completeness before scoring.

A completed run returns `files` and `coverage`, and writes `files.parquet` and `coverage.parquet` under OUTPUT. Use the returned file paths (or the path column in files.parquet) for the current selection; a recursive glob can include retained files from older plans. Failed pulls propagate errors, keep completed files resumable, and leave uncompleted coverage marked planned.


In [ ]:
from datetime import date
from comm_research.markets.weather import ecmwf

START = date(2000, 1, 1)
END = date(2026, 10, 1)  # Exclusive target-week start.
LEAD_DAYS = 26
WORKERS = 2
AS_OF = None  # Or fix a date for reproducibility.
REFERENCE_DATES = None  # Automatic; optional explicit list bypasses discovery.
CATALOGUE_CYCLES = None  # Automatic; optional IRI cycle names.
REFRESH_CATALOGUE = False
REFRESH = False
DRY_RUN = True

TARGETS = ecmwf.TARGETS
OUTPUT = ecmwf.OUTPUT
RAW = ecmwf.RAW


In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

result = ecmwf.run(targets=TARGETS, output=OUTPUT, raw=RAW, start=START, end=END,
                   lead_days=LEAD_DAYS, workers=WORKERS, as_of=AS_OF,
                   reference_dates=REFERENCE_DATES, catalogue_cycles=CATALOGUE_CYCLES,
                   refresh_catalogue=REFRESH_CATALOGUE, refresh=REFRESH, dry_run=DRY_RUN)
result if DRY_RUN else result["coverage"]
